# PGA-UNet — Cross-Validation 4 Folds

| Phần | Fold | Nội dung |
|---|---|---|
| 1 | Fold 1 | Setup (repo riêng, dataset riêng, ckpt riêng) → Test → Vis |
| 2 | Fold 2 | Setup → Test → Vis |
| 3 | Fold 3 | Setup → Test → Vis |
| 4 | Fold 4 | Setup → Test → Vis |
| **Tổng hợp** | — | Bảng mean±std × 3 mode × 6 metric + bar chart |

Chạy tuần tự: **Fold 1 Setup → Test → Vis → Fold 2 … → Tổng hợp**

> Mỗi fold: repo riêng (`pga-repo-1..4`), dataset riêng, checkpoint riêng.
> Kết quả tính ở mức **ảnh gốc** (GT union + max-merge prediction).

---
## Phần 1 — Fold 1
---

In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 1 — SETUP  (repo, dataset, checkpoint đều riêng)
# ══════════════════════════════════════════════════════
%cd /kaggle/working
import os, gdown, sys, torch, numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
REPO_1  = f'{BASE}/pga-repo-1'
DS_1_ZIP = f'{BASE}/dataset_fold1.zip'
CKPT_1  = f'{REPO_1}/checkpoints/pga_unet_expB_best.pth'

# ── Repo ──────────────────────────────────────────────────────────────
if not os.path.exists(REPO_1):
    os.system(f'git clone -q -b TN_B_ON '
              f'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git {REPO_1}')
print(f'  ✅ pga-repo-1')

# ── Dataset fold 1 ─────────────────────────────────────────────────
DS_1_ID = '1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW'
if not os.path.exists(DS_1_ZIP):
    gdown.download(f'https://drive.google.com/uc?id={DS_1_ID}', DS_1_ZIP, quiet=False)
if not os.path.exists(f'{REPO_1}/dataset_BTXRD'):
    os.system(f'unzip -oq {DS_1_ZIP} -d {REPO_1}/')
    os.system(f'rsync -a {BASE}/dataset_BTXRD/ {REPO_1}/dataset_BTXRD/ 2>/dev/null || true')
print(f'  ✅ Dataset fold 1')

# ── Checkpoint fold 1 ───────────────────────────────────────────────
CKPT_1_ID = '1MqmR4EqlWTNUzG4IAX14esKSIXT6G5cI'
os.makedirs(f'{REPO_1}/checkpoints', exist_ok=True)
if not os.path.exists(CKPT_1):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_1_ID}', CKPT_1, quiet=False)
print(f'  ✅ ckpt fold 1  {os.path.getsize(CKPT_1)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Fold 1 setup xong  |  REPO={REPO_1}')


In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 1 — TEST  (3 prompt modes, image-level merging)
# ══════════════════════════════════════════════════════
import os, csv
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

# ── Load modules từ REPO_1 ─────────────────────────────────────────
for _k in list(__import__('sys').modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del __import__('sys').modules[_k]
import sys as _sys
if REPO_1 not in _sys.path: _sys.path.insert(0, REPO_1)
else: _sys.path.remove(REPO_1); _sys.path.insert(0, REPO_1)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR_1  = f'{REPO_1}/dataset_BTXRD/test/images'
JSON_DIR_1 = f'{REPO_1}/dataset_BTXRD/test/annotations'
KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']

model_1 = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model_1.load_state_dict(torch.load(CKPT_1, map_location=DEVICE, weights_only=True))
model_1.eval()
print(f'✅ Fold 1 model loaded  [{CKPT_1}]  device={DEVICE}')

def _hd95_1(pred, gt):
    p,g=pred.astype(bool),gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe=p^binary_erosion(p); ge=g^binary_erosion(g)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def _metrics_1(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=_hd95_1(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

fold1_records = {}   # {mode: [img_rec]}
fold1_csv     = []     # [[mode, dice, iou, ...]]

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR_1, JSON_DIR_1, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i,(img_t,mask_t,prompt_t) in enumerate(tqdm(loader, desc=f'Fold1[{mode}]')):
        img_name=ds.all_samples[i][0]; gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model_1(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                   prob_max=prob.copy(),prompts=[prompt_np])
        else:
            np.maximum(groups[img_name]['gt_union'],gt_np, out=groups[img_name]['gt_union'])
            np.maximum(groups[img_name]['prob_max'],prob,  out=groups[img_name]['prob_max'])
            groups[img_name]['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=_metrics_1(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],n_samples=len(g['prompts']),**m))
    fold1_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    fold1_csv.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

del model_1
if torch.cuda.is_available(): torch.cuda.empty_cache()

bar='='*82
print(f'\n{bar}\n  FOLD 1 — PGA-UNet (GT union + max-merge)\n{bar}')
print(f"  {' Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {' N_img':>6}  {' N_smp':>6}")
print(f"  {'-'*78}")
for row in fold1_csv:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

os.makedirs(f'{REPO_1}/results', exist_ok=True)
with open(f'{REPO_1}/results/fold1_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(fold1_csv)
print(f'✅ CSV: {REPO_1}/results/fold1_results.csv')


In [ ]:
# ── Visualization Fold 1: tất cả ảnh có ≥2 GT polygon (zoom_out) ─
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert fold1_records, '❌ Chạy cell Test Fold 1 trước'

N_MIN  = 10   # tối thiểu bao nhiêu ảnh cần show
MIN_GT = 2    # số polygon tối thiểu

recs = [r for r in fold1_records['zoom_out'] if r['n_samples'] >= MIN_GT]
assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs   = recs[:max(N_MIN, len(recs))]   # show tất cả, ít nhất N_MIN
N_SHOW = len(recs)
print(f'Fold 1: {N_SHOW} ảnh có ≥{MIN_GT} GT polygon')

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Fold 1 — PGA-UNet  |  {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np        = rec['img'] * 0.5 + 0.5
    gt_np         = (rec['gt']   > 0.5).astype(float)
    pred_np       = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp = (pred_np * gt_np).sum()
    fp = (pred_np * (1 - gt_np)).sum()
    fn_ = ((1 - pred_np) * gt_np).sum()
    e   = 1e-6
    dice = float((2*tp+e) / (2*tp+fp+fn_+e))
    iou  = float((tp+e)   / (tp+fp+fn_+e))
    pre  = float((tp+e)   / (tp+fp+e))
    rec_ = float((tp+e)   / (tp+fn_+e))
    n    = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    # Col 0: ảnh gốc
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    # Col 1: ảnh + tất cả prompt (max-merged)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    # Col 2: merged prediction overlay (đỏ)
    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    # Col 3: GT union overlay (xanh lá)
    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    # Col 4: TP/FP/FN
    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np     * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1-gt_np) * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1-pred_np) * gt_np * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)


---
## Phần 2 — Fold 2
---

In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 2 — SETUP  (repo, dataset, checkpoint đều riêng)
# ══════════════════════════════════════════════════════
%cd /kaggle/working
import os, gdown, sys, torch, numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
REPO_2  = f'{BASE}/pga-repo-2'
DS_2_ZIP = f'{BASE}/dataset_fold2.zip'
CKPT_2  = f'{REPO_2}/checkpoints/pga_unet_expB_best.pth'

# ── Repo ──────────────────────────────────────────────────────────────
if not os.path.exists(REPO_2):
    os.system(f'git clone -q -b TN_B_ON '
              f'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git {REPO_2}')
print(f'  ✅ pga-repo-2')

# ── Dataset fold 2 ─────────────────────────────────────────────────
DS_2_ID = '1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW'
if not os.path.exists(DS_2_ZIP):
    gdown.download(f'https://drive.google.com/uc?id={DS_2_ID}', DS_2_ZIP, quiet=False)
if not os.path.exists(f'{REPO_2}/dataset_BTXRD'):
    os.system(f'unzip -oq {DS_2_ZIP} -d {REPO_2}/')
    os.system(f'rsync -a {BASE}/dataset_BTXRD/ {REPO_2}/dataset_BTXRD/ 2>/dev/null || true')
print(f'  ✅ Dataset fold 2')

# ── Checkpoint fold 2 ───────────────────────────────────────────────
CKPT_2_ID = '1Fu3Bfw3bdYCyCqwFhxzHb3QT3XpQmKUv'
os.makedirs(f'{REPO_2}/checkpoints', exist_ok=True)
if not os.path.exists(CKPT_2):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_2_ID}', CKPT_2, quiet=False)
print(f'  ✅ ckpt fold 2  {os.path.getsize(CKPT_2)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Fold 2 setup xong  |  REPO={REPO_2}')


In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 2 — TEST  (3 prompt modes, image-level merging)
# ══════════════════════════════════════════════════════
import os, csv
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

# ── Load modules từ REPO_2 ─────────────────────────────────────────
for _k in list(__import__('sys').modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del __import__('sys').modules[_k]
import sys as _sys
if REPO_2 not in _sys.path: _sys.path.insert(0, REPO_2)
else: _sys.path.remove(REPO_2); _sys.path.insert(0, REPO_2)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR_2  = f'{REPO_2}/dataset_BTXRD/test/images'
JSON_DIR_2 = f'{REPO_2}/dataset_BTXRD/test/annotations'
KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']

model_2 = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model_2.load_state_dict(torch.load(CKPT_2, map_location=DEVICE, weights_only=True))
model_2.eval()
print(f'✅ Fold 2 model loaded  [{CKPT_2}]  device={DEVICE}')

def _hd95_2(pred, gt):
    p,g=pred.astype(bool),gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe=p^binary_erosion(p); ge=g^binary_erosion(g)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def _metrics_2(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=_hd95_2(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

fold2_records = {}   # {mode: [img_rec]}
fold2_csv     = []     # [[mode, dice, iou, ...]]

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR_2, JSON_DIR_2, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i,(img_t,mask_t,prompt_t) in enumerate(tqdm(loader, desc=f'Fold2[{mode}]')):
        img_name=ds.all_samples[i][0]; gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model_2(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                   prob_max=prob.copy(),prompts=[prompt_np])
        else:
            np.maximum(groups[img_name]['gt_union'],gt_np, out=groups[img_name]['gt_union'])
            np.maximum(groups[img_name]['prob_max'],prob,  out=groups[img_name]['prob_max'])
            groups[img_name]['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=_metrics_2(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],n_samples=len(g['prompts']),**m))
    fold2_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    fold2_csv.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

del model_2
if torch.cuda.is_available(): torch.cuda.empty_cache()

bar='='*82
print(f'\n{bar}\n  FOLD 2 — PGA-UNet (GT union + max-merge)\n{bar}')
print(f"  {' Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {' N_img':>6}  {' N_smp':>6}")
print(f"  {'-'*78}")
for row in fold2_csv:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

os.makedirs(f'{REPO_2}/results', exist_ok=True)
with open(f'{REPO_2}/results/fold2_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(fold2_csv)
print(f'✅ CSV: {REPO_2}/results/fold2_results.csv')


In [ ]:
# ── Visualization Fold 2: tất cả ảnh có ≥2 GT polygon (zoom_out) ─
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert fold2_records, '❌ Chạy cell Test Fold 2 trước'

N_MIN  = 10   # tối thiểu bao nhiêu ảnh cần show
MIN_GT = 2    # số polygon tối thiểu

recs = [r for r in fold2_records['zoom_out'] if r['n_samples'] >= MIN_GT]
assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs   = recs[:max(N_MIN, len(recs))]   # show tất cả, ít nhất N_MIN
N_SHOW = len(recs)
print(f'Fold 2: {N_SHOW} ảnh có ≥{MIN_GT} GT polygon')

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Fold 2 — PGA-UNet  |  {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np        = rec['img'] * 0.5 + 0.5
    gt_np         = (rec['gt']   > 0.5).astype(float)
    pred_np       = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp = (pred_np * gt_np).sum()
    fp = (pred_np * (1 - gt_np)).sum()
    fn_ = ((1 - pred_np) * gt_np).sum()
    e   = 1e-6
    dice = float((2*tp+e) / (2*tp+fp+fn_+e))
    iou  = float((tp+e)   / (tp+fp+fn_+e))
    pre  = float((tp+e)   / (tp+fp+e))
    rec_ = float((tp+e)   / (tp+fn_+e))
    n    = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    # Col 0: ảnh gốc
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    # Col 1: ảnh + tất cả prompt (max-merged)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    # Col 2: merged prediction overlay (đỏ)
    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    # Col 3: GT union overlay (xanh lá)
    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    # Col 4: TP/FP/FN
    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np     * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1-gt_np) * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1-pred_np) * gt_np * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)


---
## Phần 3 — Fold 3
---

In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 3 — SETUP  (repo, dataset, checkpoint đều riêng)
# ══════════════════════════════════════════════════════
%cd /kaggle/working
import os, gdown, sys, torch, numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
REPO_3  = f'{BASE}/pga-repo-3'
DS_3_ZIP = f'{BASE}/dataset_fold3.zip'
CKPT_3  = f'{REPO_3}/checkpoints/pga_unet_expB_best.pth'

# ── Repo ──────────────────────────────────────────────────────────────
if not os.path.exists(REPO_3):
    os.system(f'git clone -q -b TN_B_ON '
              f'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git {REPO_3}')
print(f'  ✅ pga-repo-3')

# ── Dataset fold 3 ─────────────────────────────────────────────────
DS_3_ID = '1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW'
if not os.path.exists(DS_3_ZIP):
    gdown.download(f'https://drive.google.com/uc?id={DS_3_ID}', DS_3_ZIP, quiet=False)
if not os.path.exists(f'{REPO_3}/dataset_BTXRD'):
    os.system(f'unzip -oq {DS_3_ZIP} -d {REPO_3}/')
    os.system(f'rsync -a {BASE}/dataset_BTXRD/ {REPO_3}/dataset_BTXRD/ 2>/dev/null || true')
print(f'  ✅ Dataset fold 3')

# ── Checkpoint fold 3 ───────────────────────────────────────────────
CKPT_3_ID = '1eR4ULsOVbK-2GgC6SVaJqQF1et3AJ_M3'
os.makedirs(f'{REPO_3}/checkpoints', exist_ok=True)
if not os.path.exists(CKPT_3):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_3_ID}', CKPT_3, quiet=False)
print(f'  ✅ ckpt fold 3  {os.path.getsize(CKPT_3)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Fold 3 setup xong  |  REPO={REPO_3}')


In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 3 — TEST  (3 prompt modes, image-level merging)
# ══════════════════════════════════════════════════════
import os, csv
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

# ── Load modules từ REPO_3 ─────────────────────────────────────────
for _k in list(__import__('sys').modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del __import__('sys').modules[_k]
import sys as _sys
if REPO_3 not in _sys.path: _sys.path.insert(0, REPO_3)
else: _sys.path.remove(REPO_3); _sys.path.insert(0, REPO_3)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR_3  = f'{REPO_3}/dataset_BTXRD/test/images'
JSON_DIR_3 = f'{REPO_3}/dataset_BTXRD/test/annotations'
KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']

model_3 = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model_3.load_state_dict(torch.load(CKPT_3, map_location=DEVICE, weights_only=True))
model_3.eval()
print(f'✅ Fold 3 model loaded  [{CKPT_3}]  device={DEVICE}')

def _hd95_3(pred, gt):
    p,g=pred.astype(bool),gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe=p^binary_erosion(p); ge=g^binary_erosion(g)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def _metrics_3(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=_hd95_3(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

fold3_records = {}   # {mode: [img_rec]}
fold3_csv     = []     # [[mode, dice, iou, ...]]

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR_3, JSON_DIR_3, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i,(img_t,mask_t,prompt_t) in enumerate(tqdm(loader, desc=f'Fold3[{mode}]')):
        img_name=ds.all_samples[i][0]; gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model_3(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                   prob_max=prob.copy(),prompts=[prompt_np])
        else:
            np.maximum(groups[img_name]['gt_union'],gt_np, out=groups[img_name]['gt_union'])
            np.maximum(groups[img_name]['prob_max'],prob,  out=groups[img_name]['prob_max'])
            groups[img_name]['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=_metrics_3(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],n_samples=len(g['prompts']),**m))
    fold3_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    fold3_csv.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

del model_3
if torch.cuda.is_available(): torch.cuda.empty_cache()

bar='='*82
print(f'\n{bar}\n  FOLD 3 — PGA-UNet (GT union + max-merge)\n{bar}')
print(f"  {' Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {' N_img':>6}  {' N_smp':>6}")
print(f"  {'-'*78}")
for row in fold3_csv:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

os.makedirs(f'{REPO_3}/results', exist_ok=True)
with open(f'{REPO_3}/results/fold3_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(fold3_csv)
print(f'✅ CSV: {REPO_3}/results/fold3_results.csv')


In [ ]:
# ── Visualization Fold 3: tất cả ảnh có ≥2 GT polygon (zoom_out) ─
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert fold3_records, '❌ Chạy cell Test Fold 3 trước'

N_MIN  = 10   # tối thiểu bao nhiêu ảnh cần show
MIN_GT = 2    # số polygon tối thiểu

recs = [r for r in fold3_records['zoom_out'] if r['n_samples'] >= MIN_GT]
assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs   = recs[:max(N_MIN, len(recs))]   # show tất cả, ít nhất N_MIN
N_SHOW = len(recs)
print(f'Fold 3: {N_SHOW} ảnh có ≥{MIN_GT} GT polygon')

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Fold 3 — PGA-UNet  |  {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np        = rec['img'] * 0.5 + 0.5
    gt_np         = (rec['gt']   > 0.5).astype(float)
    pred_np       = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp = (pred_np * gt_np).sum()
    fp = (pred_np * (1 - gt_np)).sum()
    fn_ = ((1 - pred_np) * gt_np).sum()
    e   = 1e-6
    dice = float((2*tp+e) / (2*tp+fp+fn_+e))
    iou  = float((tp+e)   / (tp+fp+fn_+e))
    pre  = float((tp+e)   / (tp+fp+e))
    rec_ = float((tp+e)   / (tp+fn_+e))
    n    = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    # Col 0: ảnh gốc
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    # Col 1: ảnh + tất cả prompt (max-merged)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    # Col 2: merged prediction overlay (đỏ)
    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    # Col 3: GT union overlay (xanh lá)
    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    # Col 4: TP/FP/FN
    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np     * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1-gt_np) * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1-pred_np) * gt_np * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)


---
## Phần 4 — Fold 4
---

In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 4 — SETUP  (repo, dataset, checkpoint đều riêng)
# ══════════════════════════════════════════════════════
%cd /kaggle/working
import os, gdown, sys, torch, numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
REPO_4  = f'{BASE}/pga-repo-4'
DS_4_ZIP = f'{BASE}/dataset_fold4.zip'
CKPT_4  = f'{REPO_4}/checkpoints/pga_unet_expB_best.pth'

# ── Repo ──────────────────────────────────────────────────────────────
if not os.path.exists(REPO_4):
    os.system(f'git clone -q -b TN_B_ON '
              f'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git {REPO_4}')
print(f'  ✅ pga-repo-4')

# ── Dataset fold 4 ─────────────────────────────────────────────────
DS_4_ID = '1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW'
if not os.path.exists(DS_4_ZIP):
    gdown.download(f'https://drive.google.com/uc?id={DS_4_ID}', DS_4_ZIP, quiet=False)
if not os.path.exists(f'{REPO_4}/dataset_BTXRD'):
    os.system(f'unzip -oq {DS_4_ZIP} -d {REPO_4}/')
    os.system(f'rsync -a {BASE}/dataset_BTXRD/ {REPO_4}/dataset_BTXRD/ 2>/dev/null || true')
print(f'  ✅ Dataset fold 4')

# ── Checkpoint fold 4 ───────────────────────────────────────────────
CKPT_4_ID = '1Cia0-ZFGWwkraABBK-yVzHUquY52ZzxB'
os.makedirs(f'{REPO_4}/checkpoints', exist_ok=True)
if not os.path.exists(CKPT_4):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_4_ID}', CKPT_4, quiet=False)
print(f'  ✅ ckpt fold 4  {os.path.getsize(CKPT_4)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Fold 4 setup xong  |  REPO={REPO_4}')


In [ ]:
# ══════════════════════════════════════════════════════
# FOLD 4 — TEST  (3 prompt modes, image-level merging)
# ══════════════════════════════════════════════════════
import os, csv
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

# ── Load modules từ REPO_4 ─────────────────────────────────────────
for _k in list(__import__('sys').modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del __import__('sys').modules[_k]
import sys as _sys
if REPO_4 not in _sys.path: _sys.path.insert(0, REPO_4)
else: _sys.path.remove(REPO_4); _sys.path.insert(0, REPO_4)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR_4  = f'{REPO_4}/dataset_BTXRD/test/images'
JSON_DIR_4 = f'{REPO_4}/dataset_BTXRD/test/annotations'
KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']

model_4 = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model_4.load_state_dict(torch.load(CKPT_4, map_location=DEVICE, weights_only=True))
model_4.eval()
print(f'✅ Fold 4 model loaded  [{CKPT_4}]  device={DEVICE}')

def _hd95_4(pred, gt):
    p,g=pred.astype(bool),gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe=p^binary_erosion(p); ge=g^binary_erosion(g)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def _metrics_4(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=_hd95_4(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

fold4_records = {}   # {mode: [img_rec]}
fold4_csv     = []     # [[mode, dice, iou, ...]]

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR_4, JSON_DIR_4, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i,(img_t,mask_t,prompt_t) in enumerate(tqdm(loader, desc=f'Fold4[{mode}]')):
        img_name=ds.all_samples[i][0]; gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model_4(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                   prob_max=prob.copy(),prompts=[prompt_np])
        else:
            np.maximum(groups[img_name]['gt_union'],gt_np, out=groups[img_name]['gt_union'])
            np.maximum(groups[img_name]['prob_max'],prob,  out=groups[img_name]['prob_max'])
            groups[img_name]['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=_metrics_4(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],n_samples=len(g['prompts']),**m))
    fold4_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    fold4_csv.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

del model_4
if torch.cuda.is_available(): torch.cuda.empty_cache()

bar='='*82
print(f'\n{bar}\n  FOLD 4 — PGA-UNet (GT union + max-merge)\n{bar}')
print(f"  {' Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {' N_img':>6}  {' N_smp':>6}")
print(f"  {'-'*78}")
for row in fold4_csv:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

os.makedirs(f'{REPO_4}/results', exist_ok=True)
with open(f'{REPO_4}/results/fold4_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(fold4_csv)
print(f'✅ CSV: {REPO_4}/results/fold4_results.csv')


In [ ]:
# ── Visualization Fold 4: tất cả ảnh có ≥2 GT polygon (zoom_out) ─
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert fold4_records, '❌ Chạy cell Test Fold 4 trước'

N_MIN  = 10   # tối thiểu bao nhiêu ảnh cần show
MIN_GT = 2    # số polygon tối thiểu

recs = [r for r in fold4_records['zoom_out'] if r['n_samples'] >= MIN_GT]
assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs   = recs[:max(N_MIN, len(recs))]   # show tất cả, ít nhất N_MIN
N_SHOW = len(recs)
print(f'Fold 4: {N_SHOW} ảnh có ≥{MIN_GT} GT polygon')

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'Fold 4 — PGA-UNet  |  {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np        = rec['img'] * 0.5 + 0.5
    gt_np         = (rec['gt']   > 0.5).astype(float)
    pred_np       = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp = (pred_np * gt_np).sum()
    fp = (pred_np * (1 - gt_np)).sum()
    fn_ = ((1 - pred_np) * gt_np).sum()
    e   = 1e-6
    dice = float((2*tp+e) / (2*tp+fp+fn_+e))
    iou  = float((tp+e)   / (tp+fp+fn_+e))
    pre  = float((tp+e)   / (tp+fp+e))
    rec_ = float((tp+e)   / (tp+fn_+e))
    n    = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    # Col 0: ảnh gốc
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    # Col 1: ảnh + tất cả prompt (max-merged)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    # Col 2: merged prediction overlay (đỏ)
    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    # Col 3: GT union overlay (xanh lá)
    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    # Col 4: TP/FP/FN
    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np     * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1-gt_np) * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1-pred_np) * gt_np * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)


---
## Tổng hợp — Cross-Validation 4 Folds
---

In [ ]:
# ══════════════════════════════════════════════════════
# TỔNG HỢP — Cross-Validation 4 Folds (PGA-UNet)
# fold1_csv, fold2_csv, fold3_csv, fold4_csv phải đã chạy
# ══════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt, csv, os
from IPython.display import display as _d

KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']
MODE_LABELS = ['Zoom-Out','Shift','Mixed (70/30)']
FOLD_LABELS = ['Fold 1','Fold 2','Fold 3','Fold 4']
FOLD_COLORS = ['#1565C0','#2E7D32','#F57F17','#C62828']

# Gom csv_rows từ 4 fold
# csv row format: [mode, dice, iou, prec, rec, hd95, cbl, N_img, N_smp]
all_csvs = {1: fold1_csv, 2: fold2_csv, 3: fold3_csv, 4: fold4_csv}

# {fold_num: {mode: {metric: value}}}
fold_mode_avg = {}
for fn, rows in all_csvs.items():
    fold_mode_avg[fn] = {}
    for row in rows:
        mode = row[0]
        fold_mode_avg[fn][mode] = {k: float(row[i+1]) for i,k in enumerate(KEYS)}

# ── Bảng tổng hợp ────────────────────────────────────────────────────
bar = '═'*96
print(f'\n{bar}')
print('  CROSS-VALIDATION 4 FOLDS — PGA-UNet  (image-level GT union + max-merge)')
print(f'{bar}')
print(f"  {'Fold':<8} {'Mode':<13}"+''.join(f'{h:>9}' for h in HDRS))
print('  '+'-'*90)

all_mode_vals = {m: {k:[] for k in KEYS} for m in MODES}
for fn in [1,2,3,4]:
    for mode in MODES:
        m = fold_mode_avg[fn][mode]
        print(f"  Fold {fn:<4} {mode:<13}"+''.join(f'{m[k]:>9.4f}' for k in KEYS))
        for k in KEYS: all_mode_vals[mode][k].append(m[k])
    print('  '+'-'*90)

print(f"\n  {'Mean':<8} {'Mode':<13}"+''.join(f'{h:>9}' for h in HDRS))
print('  '+'-'*90)
avg_results = {}
for mode in MODES:
    avg_results[mode] = {}
    mu_row = []; sd_row = []
    for k in KEYS:
        mu = np.mean(all_mode_vals[mode][k])
        sd = np.std(all_mode_vals[mode][k])
        avg_results[mode][k] = (mu, sd)
        mu_row.append(f'{mu:.4f}'); sd_row.append(f'±{sd:.4f}')
    print(f"  {'mean':<8} {mode:<13}"+''.join(f'{p:>9}' for p in mu_row))
    print(f"  {'±std':<8} {'':<13}"+''.join(f'{p:>9}' for p in sd_row))
    print('  '+'-'*90)
print(bar)

# ── CSV tổng hợp ─────────────────────────────────────────────────────
BASE = '/kaggle/working' if __import__('os').path.exists('/kaggle/working') else '/content'
os.makedirs(f'{BASE}/results_cv4', exist_ok=True)
cv_csv = f'{BASE}/results_cv4/cv4_summary.csv'
with open(cv_csv,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['fold','mode']+KEYS)
    for fn in [1,2,3,4]:
        for mode in MODES:
            m=fold_mode_avg[fn][mode]
            w.writerow([f'fold{fn}',mode]+[f'{m[k]:.4f}' for k in KEYS])
    w.writerow([])
    for mode in MODES:
        w.writerow(['mean',mode]+[f'{avg_results[mode][k][0]:.4f}' for k in KEYS])
        w.writerow(['std', mode]+[f'{avg_results[mode][k][1]:.4f}' for k in KEYS])
print(f'✅ CSV: {cv_csv}')

# ── Biểu đồ cột ──────────────────────────────────────────────────────
METRIC_PAIRS = [
    ('dice',      'Dice ↑'),
    ('iou',       'IoU ↑'),
    ('precision', 'Precision ↑'),
    ('recall',    'Recall ↑'),
    ('hd95',      'HD95 ↓ (px)'),
    ('cbl',       'CBL ↑'),
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('PGA-UNet — Cross-Validation 4 Folds  (Image-level, IMG_SIZE=512)',
             fontsize=14, fontweight='bold', y=1.01)
n_modes = len(MODES)
x = np.arange(n_modes)
w_bar = 0.18

for ax_idx, (metric, label) in enumerate(METRIC_PAIRS):
    ax = axes[ax_idx]
    for fi, fn in enumerate([1,2,3,4]):
        vals = [fold_mode_avg[fn][m][metric] for m in MODES]
        offset = (fi - 1.5) * w_bar
        bars = ax.bar(x+offset, vals, w_bar, color=FOLD_COLORS[fi], alpha=0.85,
                      edgecolor='white', label=FOLD_LABELS[fi] if ax_idx==0 else '_')
        for b,v in zip(bars,vals):
            fmt = f'{v:.0f}' if metric=='hd95' else f'{v:.3f}'
            ax.text(b.get_x()+b.get_width()/2,
                    v+(0.005 if metric!='hd95' else 0.5),
                    fmt, ha='center', va='bottom', fontsize=6.5,
                    fontweight='bold', color=FOLD_COLORS[fi])
    # Đường mean
    mu_vals = [avg_results[m][metric][0] for m in MODES]
    ax.plot(x, mu_vals, 'k--o', linewidth=1.8, markersize=5,
            label='Mean' if ax_idx==0 else '_', zorder=5)
    for xi, mv in zip(x, mu_vals):
        fmt = f'{mv:.0f}' if metric=='hd95' else f'{mv:.3f}'
        ax.text(xi, mv+(0.016 if metric!='hd95' else 1.5), fmt,
                ha='center', va='bottom', fontsize=7.5, fontweight='bold', color='black')
    ax.set_title(label, fontsize=12, fontweight='bold', pad=8)
    all_vals = [fold_mode_avg[fn][m][metric] for fn in [1,2,3,4] for m in MODES]
    hi = max(all_vals)
    ax.set_ylim(0, hi*(1.25 if metric!='hd95' else 1.35))
    ax.set_xticks(x); ax.set_xticklabels(MODE_LABELS, fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

handles = [plt.Rectangle((0,0),1,1,color=c,alpha=0.85) for c in FOLD_COLORS]
handles += [plt.Line2D([0],[0],color='black',linestyle='--',marker='o',linewidth=1.8,markersize=5)]
fig.legend(handles, FOLD_LABELS+['Mean'],
           loc='lower center', ncol=5, fontsize=10, bbox_to_anchor=(0.5,-0.02))
plt.tight_layout()
plot_path = f'{BASE}/results_cv4/cv4_comparison.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
_d(fig); plt.close(fig)
print(f'✅ Biểu đồ: {plot_path}')
